# Data Cleaning


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import hashlib
raw_data_path= '../Data/Raw Data/'
clean_data_path= '../Data/Clean Data/'
raw_data_file=raw_data_path+'raw_data.csv'
df=pd.read_csv(raw_data_file)

In [ ]:
# Rename columns for convenience
column_names={
    'TRANSACTION_NUMBER':'transaction_num',
    'INSTANCE_DATE':'instance_date',
    'GROUP_EN':'group',
    'PROCEDURE_EN':'procedure',
    'IS_OFFPLAN_EN':'is_offplan',
    'IS_FREE_HOLD_EN':'is_free_hold',
    'USAGE_EN':'usage',
    'AREA_EN':'area',
    'PROP_TYPE_EN':'prop_type',
    'PROP_SB_TYPE_EN':'prop_sb_type',
    'TRANS_VALUE':'trans_value',
    'PROCEDURE_AREA':'procedure_area',
    'ACTUAL_AREA':'actual_area',
    'ROOMS_EN':'rooms',
    'PARKING':'parking',
    'NEAREST_METRO_EN':'nearest_metro',
    'NEAREST_MALL_EN':'nearest_mall',
    'NEAREST_LANDMARK_EN':'nearest_landmark',
    'TOTAL_BUYER':'total_buyer',
    'TOTAL_SELLER':'total_seller',
    'MASTER_PROJECT_EN':'master_project',
    'PROJECT_EN':'project'
}
df.rename(columns=column_names,inplace=True)

#Hash transaction number
df['transaction_num']=df['transaction_num'].apply(lambda x:hashlib.sha256(str(x).encode()).hexdigest())
#Drop requried columns
df=df.drop(columns=['total_buyer','total_seller','master_project'])

In [ ]:
#Turn date datatype to datetime
df['instance_date']=pd.to_datetime(df['instance_date'])
df['date_year']= df['instance_date'].dt.year
#Months turned into sin and cos vectors
df['month_sin']=np.sin(2 * np.pi*df['instance_date'].dt.month/12.0)
df['month_cos']=np.cos(2*np.pi*df['instance_date'].dt.month/12.0)

In [ ]:
#Calculate the baseline price per square foot
df['price_per_sqft']=df['trans_value']/df['actual_area']

#Sort chronologically within each community to properly calculate rolling windows
df=df.sort_values(by=['area','instance_date'])

#Set the date as the index for time based rolling math
df.set_index('instance_date',inplace=True)

#Generate the rolling 30 day average price per square foot for each specific area
df['community_30d_avg_price']=df.groupby('area')['price_per_sqft'].transform(lambda x:x.rolling('30D').mean())

#Reset the index and drop the raw timestamp now that temporal features are extracted
df.reset_index(inplace=True)
df=df.drop(columns=['instance_date'])

#Forward fill any nan values created by the first 30 days of the dataset
df['community_30d_avg_price']=df['community_30d_avg_price'].bfill()

In [ ]:
#Clean up rooms values
df['rooms']=df['rooms'].astype(str).str.extract(r'(\d+)').fillna(0).astype(int)

#Clean parking by finding first number found. if not found return nan
def clean_parking(val):
    if pd.isna(val): return np.nan
    val=str(val)
    import re
    nums=re.findall(r'\d+', val)
    if nums: return int(nums[0])
    return np.nan
df['parking']=df['parking'].apply(clean_parking)

In [ ]:
df['is_offplan']=df['is_offplan'].map({'Off-Plan':1, 'Ready':0})
df['is_free_hold']=df['is_free_hold'].map({'Free Hold':1,'Non Free Hold':0})
#For unexpected values
df['is_offplan']=df['is_offplan'].fillna(0).astype(int)
df['is_free_hold']=df['is_free_hold'].fillna(0).astype(int)

In [ ]:
#Fill categorical columns with Unknown
cat_cols_with_na=['prop_sb_type','nearest_metro','nearest_mall','nearest_landmark','project']
df[cat_cols_with_na]=df[cat_cols_with_na].fillna('Unknown')
#Fill numerical columns with median
num_cols_with_na=['rooms','parking']
for col in num_cols_with_na:
    df[col]=df[col].fillna(df[col].median())

In [ ]:
#Remove rows with negative or zero area/value
df = df[df['trans_value']>0]
df = df[df['actual_area']>0]
df = df[df['procedure_area']>0]

#Remove extreme top 1% outliers for numerical features
for col in ['trans_value','actual_area','procedure_area']:
    q99=df[col].quantile(0.99)
    df=df[df[col]<=q99]

In [ ]:
#Binary encoding for columns with binary values
binary_cols=['is_offplan','is_free_hold','usage']
for col in binary_cols:
    df[col]=LabelEncoder().fit_transform(df[col].astype(str))

area_target_mean=df.groupby('area')['price_per_sqft'].mean()
df['area']=df['area'].map(area_target_mean)

# Label encode the columns with many values
high_card_cols=['project','nearest_metro','nearest_mall','nearest_landmark','prop_sb_type']
for col in high_card_cols:
    df[col]=LabelEncoder().fit_transform(df[col].astype(str))

# One hot encode columns with a small number of values more than 2
low_card_cols=['group','procedure','prop_type']
df=pd.get_dummies(df,columns=low_card_cols,drop_first=True)

In [10]:
df.to_csv(clean_data_path + 'cleaned_data.csv', index=False)

In [ ]:
dfc=pd.read_csv(clean_data_path+'cleaned_data.csv')
dfc.head()
for col in dfc.columns:
    print(f"{col}: {len(df[col].tolist())}")

transaction_num: 26981
is_offplan: 26981
is_free_hold: 26981
usage: 26981
area: 26981
prop_sb_type: 26981
trans_value: 26981
procedure_area: 26981
actual_area: 26981
rooms: 26981
parking: 26981
nearest_metro: 26981
nearest_mall: 26981
nearest_landmark: 26981
project: 26981
date_year: 26981
Month_sin: 26981
Month_cos: 26981
price_per_sqft: 26981
community_30d_avg_price: 26981
group_Mortgage: 26981
group_Sales: 26981
procedure_Delayed Mortgage: 26981
procedure_Delayed Sell: 26981
procedure_Delayed Sell Lease to Own Registration: 26981
procedure_Development Mortgage: 26981
procedure_Development Registration: 26981
procedure_Development Registration Pre-Registration: 26981
procedure_Grant: 26981
procedure_Grant Development: 26981
procedure_Grant Pre-Registration: 26981
procedure_Grant on Delayed Sell: 26981
procedure_Lease Finance Modification: 26981
procedure_Lease Finance Registration: 26981
procedure_Lease to Own Modify: 26981
procedure_Lease to Own Registration: 26981
procedure_Lease t